<img src="http://hilpisch.com/tpq_logo.png" alt="The Python Quants" width="35%" align="right" border="0"><br>

# Python for Algorithmic Trading

**Chapter 09 &mdash; FX Trading with FXCM (Server, Keras Training Example)**

# This notebook is based on the fxcmpy.py module from FXCM. Unfortunately, FXCM has discontinued support and distribution of fxcmpy.py, so this notebook is no longer functional.

## Risk Disclaimer

<font size="-1">
Trading forex/CFDs on margin carries a high level of risk and may not be suitable for all investors as you could sustain losses in excess of deposits. Leverage can work against you. Due to the certain restrictions imposed by the local law and regulation, German resident retail client(s) could sustain a total loss of deposited funds but are not subject to subsequent payment obligations beyond the deposited funds. Be aware and fully understand all risks associated with the market and trading. Prior to trading any products, carefully consider your financial situation and experience level. Any opinions, news, research, analyses, prices, or other information is provided as general market commentary, and does not constitute investment advice. FXCM & TPQ will not accept liability for any loss or damage, including without limitation to, any loss of profit, which may arise directly or indirectly from use of or reliance on such information.
</font>

## Author Disclaimer

The author is neither an employee, agent nor representative of FXCM and is therefore acting independently. The opinions given are their own, constitute general market commentary, and do not constitute the opinion or advice of FXCM or any form of personal or investment advice. FXCM assumes no responsibility for any loss or damage, including but not limited to, any loss or gain arising out of the direct or indirect use of this or any other content. Trading forex/CFDs on margin carries a high level of risk and may not be suitable for all investors as you could sustain losses in excess of deposits.

In [ ]:
!git clone https://github.com/tpq-classes/python_for_algo_trading_core.git
import sys
sys.path.append('python_for_algo_trading_core')


In [ ]:
import time
import numpy as np
import pandas as pd
import datetime as dt
from pylab import mpl, plt

In [ ]:
plt.style.use('seaborn-v0_8')
mpl.rcParams['font.family'] = 'serif'
%matplotlib inline

## Connecting to the API

In [ ]:
import fxcmpy

In [ ]:
fxcmpy.__version__

In [ ]:
%time api = fxcmpy.fxcmpy(config_file='../pyalgo.cfg')

In [ ]:
instruments = api.get_instruments()

In [ ]:
print(instruments)

## Retrieving Historical Data

In [ ]:
candles = api.get_candles('BTC/USD', period='D1', number=10)

In [ ]:
candles[candles.columns[:4]]

In [ ]:
candles[candles.columns[4:]]

In [ ]:
start = dt.datetime(2019, 1, 1)
end = dt.datetime(2020, 7, 28)

In [ ]:
candles = api.get_candles('BTC/USD', period='D1',
                          start=start, stop=end)

In [ ]:
candles.info()

The parameter `period` must be one of `m1, m5, m15, m30, H1, H2, H3, H4, H6, H8, D1, W1` or `M1`.

In [ ]:
candles = api.get_candles('ETH/USD', period='m1', number=250)

In [ ]:
candles

In [ ]:
candles['askclose'].plot(figsize=(10, 6));

## Streaming Data

In [ ]:
def output(data, dataframe):
    print('%3d | %s | %s | %6.5f, %6.5f'
          % (len(dataframe), data['Symbol'],
             pd.to_datetime(int(data['Updated']), unit='ms'),
             data['Rates'][0], data['Rates'][1]))

In [ ]:
api.subscribe_market_data('BTC/USD', (output,))

In [ ]:
api.get_last_price('BTC/USD')

In [ ]:
api.unsubscribe_market_data('BTC/USD')

## Placing Orders

In [ ]:
api.get_open_positions()

In [ ]:
order = api.create_market_buy_order('BTC/USD', 500)

In [ ]:
sel = ['tradeId', 'amountK', 'currency',
       'grossPL', 'isBuy']

In [ ]:
api.get_open_positions()[sel]

In [ ]:
order = api.create_market_buy_order('ETH/USD', 500)

In [ ]:
api.get_open_positions()[sel]

In [ ]:
order = api.create_market_sell_order('BTC/USD', 250)

In [ ]:
order = api.create_market_buy_order('ETH/USD', 500)

In [ ]:
api.get_open_positions()[sel]

In [ ]:
api.close_all_for_symbol('ETH/USD')

In [ ]:
api.get_open_positions()[sel]

In [ ]:
api.close_all()

In [ ]:
api.get_open_positions()

## Account Information

In [ ]:
api.get_default_account()

In [ ]:
api.get_accounts().T

## Deep Learning Strategy

In [ ]:
import tensorflow as tf
from keras.layers import Dense
from keras.models import Sequential

In [ ]:
candles = api.get_candles('BTC/USD', period='H1', number=2500)

In [ ]:
candles.head()

In [ ]:
symbol = 'BTC/USD'

In [ ]:
data = pd.DataFrame((candles['bidclose'] + candles['askclose']) / 2,
                    index=candles.index, columns=[symbol])

In [ ]:
data.info()

In [ ]:
data['r'] = np.log(data / data.shift(1))

In [ ]:
data['d'] = np.where(data['r'] > 0, 1, 0)

In [ ]:
lags = 3
cols = list()
for f in ['r']:
#for f in [symbol]:
#for f in [symbol, 'r']:
    for lag in range(1, lags + 1):
        col = f'{f}_lag_{lag}'
        data[col] = data[f].shift(lag)
        cols.append(col)

In [ ]:
data.dropna(inplace=True)

In [ ]:
data.head()

In [ ]:
split = int(len(data) * 0.75)

In [ ]:
train = data.iloc[:split].copy()

In [ ]:
mu, std = train.mean(), train.std()

In [ ]:
train_ = (train - mu) / std

In [ ]:
train_[cols].head()

In [ ]:
test = data.iloc[split:].copy()

In [ ]:
test_ = (test - mu) / std

In [ ]:
np.random.seed(100)
tf.random.set_seed(100)

In [ ]:
model = Sequential()
model.add(Dense(64, input_dim=len(cols), activation='relu'))
# model.add(Dense(32, activation='relu'))
model.add(Dense(1, activation='sigmoid'))
model.compile(loss='binary_crossentropy',
              optimizer='adam', metrics=['accuracy'])

In [ ]:
model.summary()

In [ ]:
%%time
model.fit(train_[cols], train['d'],
        epochs=75, verbose=False,
        validation_split=0.1, shuffle=False)

In [ ]:
model.evaluate(train_[cols], train['d'])

In [ ]:
model.evaluate(test_[cols], test['d'])

In [ ]:
test['p'] = model.predict_classes(test_[cols])

In [ ]:
test['p'].value_counts()

In [ ]:
test['p'] = np.where(test['p'] == 1, 1, -1)

In [ ]:
test['p'].value_counts()

In [ ]:
test['s'] = test['p'] * test['r']

In [ ]:
test[['r', 's']].sum().apply(np.exp)

In [ ]:
test[['r', 's']].cumsum().apply(np.exp).plot(figsize=(10, 6));

<img src="http://hilpisch.com/tpq_logo.png" alt="The Python Quants" width="35%" align="right" border="0"><br>

<a href="http://tpq.io" target="_blank">http://tpq.io</a> | <a href="http://twitter.com/dyjh" target="_blank">@dyjh</a> | <a href="mailto:training@tpq.io">training@tpq.io</a>